# Noble AI — Google Colab P0 / pruebas reales de entorno

**Objetivo:** documentar hardware, comprobar la integridad estructural del repositorio y realizar opcionalmente una **inferencia real** con un modelo pequeño. **Este cuaderno no es la aplicación Noble AI** y NO certifica la integración con KoboldCpp/ComfyUI/TTS ni el perfil de 12/16 GB.

Instrucciones: 1) En Colab selecciona GPU si hay disponible; 2) Ejecuta todas las celdas; 3) Guarda el JSON final para añadir resultados a `docs/PROGRESO.md`. No uses túneles/anti-AFK ni web UIs públicas en runtimes gratuitos. Fuente: https://research.google.com/colaboratory/intl/es/faq.html

Modelos: prueba rápida por defecto `Qwen/Qwen3-1.7B` (no representa calidad final), candidato final Qwen3-8B GGUF para 12/16GB, por evaluar en fase P1. La inferencia solo se intenta si se detecta GPU.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, platform, time, shutil
ROOT = Path("/content/novel_ai") if Path("/content").exists() else Path.cwd() / "novel_ai"
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Mairon31/novel_ai.git", str(ROOT)], check=True)
else:
    print("Clon existente; se respetan los archivos locales:", ROOT)
commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Repositorio:", ROOT)
print("Commit:", commit)
RESULT = {"project":"Noble AI","phase":"P0","commit":commit,"environment":{},"checks":{},"timestamp_utc":__import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat()}

In [ ]:
import importlib.util
has_torch = importlib.util.find_spec("torch") is not None
if has_torch:
    import torch
    cuda = bool(torch.cuda.is_available())
    gpu = torch.cuda.get_device_name(0) if cuda else None
    total = int(torch.cuda.get_device_properties(0).total_memory / 1024**2) if cuda else None
else:
    cuda, gpu, total = False, None, None
diagnostics = {
 "python":platform.python_version(),"platform":platform.platform(),
 "torch":torch.__version__ if has_torch else None,
 "torch_cuda":torch.version.cuda if has_torch else None,
 "gpu":gpu,"vram_total_mb":total,
 "disk_free_gb":round(shutil.disk_usage(str(ROOT)).free / 1024**3,2)
}
if shutil.which("nvidia-smi"):
    p=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free,driver_version","--format=csv,noheader"],capture_output=True,text=True)
    diagnostics["nvidia_smi"]=p.stdout.strip() if p.returncode==0 else "nvidia-smi failed"
RESULT["environment"]=diagnostics
RESULT["checks"]["COLAB_DIAG"]={"status":"PASS","details":"Entorno detectado; GPU no garantizada"}
print(json.dumps(diagnostics,ensure_ascii=False,indent=2))

In [ ]:
tests_dir=ROOT/"tests"
if tests_dir.exists():
    run=subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p","test_*.py"],cwd=ROOT,capture_output=True,text=True,timeout=120)
    RESULT["checks"]["DOC_CONTRACT"]={"status":"PASS" if run.returncode==0 else "FAIL","returncode":run.returncode,"stdout_tail":run.stdout[-500:],"stderr_tail":run.stderr[-2000:]}
    print(run.stdout[-2000:]); print(run.stderr[-2000:]); print("Return code:",run.returncode)
else:
    RESULT["checks"]["DOC_CONTRACT"]={"status":"SKIP","reason":"tests/ aún no existe"}
    print("SKIP: no hay tests")

## Inferencia de modelo real (opcional)

Por defecto se intenta con GPU disponible. Se descarga un modelo de **1.7B** como prueba técnica del entorno, NO como prueba de narración de producción. Puede consumir varios GB de descarga. Si Colab no asigna GPU, la prueba queda en SKIP. El backend GGUF KoboldCpp se verificará en una fase separada.

In [ ]:
RUN_LLM_SMOKE = True
MODEL_ID = "Qwen/Qwen3-1.7B"
if RUN_LLM_SMOKE and cuda:
    import traceback
    start=time.time()
    try:
        subprocess.run([sys.executable,"-m","pip","-q","install","-U","transformers>=4.51,<5","accelerate"],check=True,timeout=600)
        from transformers import AutoTokenizer, AutoModelForCausalLM
        tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
        model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16,device_map="auto")
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        messages=[
            {"role":"system","content":"Escribe narrativa breve de fantasía en español latinoamericano. No tomes decisiones por el jugador."},
            {"role":"user","content":"La protagonista entra a una biblioteca encantada. Narra dos frases con una decisión pendiente."}
        ]
        inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,enable_thinking=False,return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated=model.generate(inputs,max_new_tokens=110,do_sample=True,temperature=0.7,top_p=0.8,pad_token_id=tokenizer.eos_token_id)
        narrative=tokenizer.decode(generated[0][inputs.shape[-1]:],skip_special_tokens=True).strip()
        if not narrative: raise ValueError("El modelo devolvió texto vacío")
        RESULT["checks"]["COLAB_LLM_SMALL_REAL"]={"status":"PASS","model":MODEL_ID,"seconds":round(time.time()-start,2),"peak_torch_allocated_mb":round(torch.cuda.max_memory_allocated()/1024**2),"sample":narrative[:600]}
        print("Generación real:",narrative)
        del model,tokenizer,generated,inputs
        import gc; gc.collect(); torch.cuda.empty_cache()
    except Exception as exc:
        RESULT["checks"]["COLAB_LLM_SMALL_REAL"]={"status":"FAIL","model":MODEL_ID,"error":repr(exc),"traceback_tail":traceback.format_exc()[-1600:]}
        print("FALLO de inferencia:",repr(exc))
else:
    RESULT["checks"]["COLAB_LLM_SMALL_REAL"]={"status":"SKIP","reason":"GPU no disponible o RUN_LLM_SMOKE=False"}
    print("SKIP: no se ha cargado un LLM real")

## Estado de proveedores finales

Esta primera versión solo comprueba condiciones para comenzar. **No** instala por sí sola motores de difusión/TTS ni abre túneles. Los gates `COLAB_LLM_GGUF_REAL`, `COLAB_IMAGE_REAL`, `COLAB_TTS_MANUAL` permanecerán NOT_RUN hasta contar con adaptadores y pruebas específicas.

In [ ]:
for name in ("COLAB_LLM_GGUF_REAL","COLAB_IMAGE_REAL","COLAB_TTS_MANUAL","WINDOWS_12GB","WINDOWS_16GB"):
    RESULT["checks"][name]={"status":"NOT_RUN","reason":"Fase de integración pendiente"}
result_path=ROOT/"noble_ai_probe_results.json"
result_path.write_text(json.dumps(RESULT,ensure_ascii=False,indent=2),encoding="utf-8")
print("Guardado:",result_path)
print(json.dumps({k:v.get("status") for k,v in RESULT["checks"].items()},ensure_ascii=False,indent=2))
print("Guardar este JSON como evidencia antes de reiniciar Colab; nunca subir al repositorio historias ni secretos.")